# Data Cleaning
**This notebook prepares Flatiron Health CSV files for patients with advanced urothelial cancer in the test set. For details on training and test split, see notebook "data_cleaning_training.ipynb."**

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorUrothelial
from flatiron_cleaner import merge_dataframes

## Import data 

In [2]:
test = pd.read_csv('../outputs/test_patient_ids.csv')

In [3]:
test.shape

(2626, 1)

In [4]:
test.head(3)

,PatientID
0,F75AB80F63FFB
1,F64FAA25D164C
2,FC0DEB3C3CDEA


In [5]:
test_ids = test.PatientID.to_list()

## Data cleaning 

In [6]:
# Initialize class 
processor = DataProcessorUrothelial()

In [7]:
# Index date dataframe
df = pd.read_csv('../data/Enhanced_AdvUrothelial.csv')
df = (
    df
    .query('PatientID in @test_ids')
    [['PatientID', 'AdvancedDiagnosisDate']]
)

### Process Enhanced_AdvUrothelial.csv

In [8]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvUrothelial.csv',
                                         patient_ids = test_ids)

2025-04-26 12:20:26,829 - INFO - Successfully read Enhanced_AdvUrothelial.csv file with shape: (13129, 13) and unique PatientIDs: 13129
2025-04-26 12:20:26,829 - INFO - Filtering for 2626 specific PatientIDs
2025-04-26 12:20:26,831 - INFO - Successfully filtered Enhanced_AdvUrothelial.csv file with shape: (2626, 13) and unique PatientIDs: 2626
2025-04-26 12:20:26,842 - INFO - Successfully processed Enhanced_AdvUrothelial.csv file with final shape: (2626, 13) and unique PatientIDs: 2626


In [9]:
enhanced_df['SmokingStatus'] = enhanced_df['SmokingStatus'].map({
    'History of smoking': 1,
    'No history of smoking': 0,
    'Unknown/not documented': 0
})

In [10]:
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)
enhanced_df['days_diagnosis_to_surgery'] = enhanced_df['days_diagnosis_to_surgery'].fillna(0)

In [11]:
enhanced_df['SurgeryType_mod'] = enhanced_df['SurgeryType_mod'].fillna('unknown')

### Process Demographics.csv 

In [12]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = df,
                                                 index_date_column = 'AdvancedDiagnosisDate')

2025-04-26 12:20:26,867 - INFO - Successfully read Demographics.csv file with shape: (13129, 6) and unique PatientIDs: 13129
2025-04-26 12:20:26,876 - INFO - Successfully processed Demographics.csv file with final shape: (2626, 6) and unique PatientIDs: 2626


In [13]:
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [14]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvUrothelialBiomarkers.csv

In [15]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvUrothelialBiomarkers.csv',
                                             index_date_df = df, 
                                             index_date_column = 'AdvancedDiagnosisDate',
                                             days_before = None, 
                                             days_after = 14)

2025-04-26 12:20:26,901 - INFO - Successfully read Enhanced_AdvUrothelialBiomarkers.csv file with shape: (9924, 19) and unique PatientIDs: 4251
2025-04-26 12:20:26,908 - INFO - Successfully merged Enhanced_AdvUrothelialBiomarkers.csv df with index_date_df resulting in shape: (2007, 20) and unique PatientIDs: 848
2025-04-26 12:20:26,922 - INFO - Successfully processed Enhanced_AdvUrothelialBiomarkers.csv file with final shape: (2626, 4) and unique PatientIDs: 2626


In [16]:
biomarkers_df.PDL1_percent_staining.value_counts(dropna = False)

PDL1_percent_staining
NaN          2619
50% - 59%       3
2% - 4%         1
5% - 9%         1
30% - 39%       1
40% - 49%       1
0%              0
< 1%            0
1%              0
10% - 19%       0
20% - 29%       0
60% - 69%       0
70% - 79%       0
80% - 89%       0
90% - 99%       0
100%            0
Name: count, dtype: int64

In [17]:
def map_pdl1(value):
    if pd.isna(value):  # leave missing as is
        return value
    elif value in ['0%', '< 1%']:
        return '0%'
    else:
        return '>=1%'

biomarkers_df['PDL1_binary'] = biomarkers_df['PDL1_percent_staining'].apply(map_pdl1)

In [18]:
biomarkers_df.PDL1_binary.value_counts(dropna = False)

PDL1_binary
NaN     2619
>=1%       7
Name: count, dtype: int64

In [19]:
biomarkers_df = biomarkers_df.drop(columns = ['PDL1_percent_staining'])

In [20]:
biomarkers_df.FGFR_status.value_counts(dropna = False)

FGFR_status
NaN         2560
negative      41
positive      25
Name: count, dtype: int64

In [21]:
biomarkers_df['FGFR_status'] = biomarkers_df['FGFR_status'].cat.add_categories('unknown').fillna('unknown')
biomarkers_df['PDL1_status'] = biomarkers_df['PDL1_status'].fillna('unknown')

### Process ECOG.csv

In [22]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = df,
                                 index_date_column = 'AdvancedDiagnosisDate',
                                 days_before = 90,
                                 days_after = 14,
                                 days_before_further = 180)

2025-04-26 12:20:27,012 - INFO - Successfully read ECOG.csv file with shape: (184794, 4) and unique PatientIDs: 9933
2025-04-26 12:20:27,041 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (38176, 5) and unique PatientIDs: 2009
2025-04-26 12:20:27,063 - INFO - Successfully processed ECOG.csv file with final shape: (2626, 3) and unique PatientIDs: 2626


In [23]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].cat.add_categories('unknown').fillna('unknown')

ecog_df['ecog_index'] = ecog_df["ecog_index"].map({
    0: '0-1',
    1: '0-1',
    2: '2',
    3: '3-4',
    4: '3-4',
    'unknown': 'unknown'
})

ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('category')

In [24]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [25]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = df,
                                     index_date_column = 'AdvancedDiagnosisDate',
                                     weight_days_before = 90,
                                     days_after = 14,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2025-04-26 12:20:30,630 - INFO - Successfully read Vitals.csv file with shape: (3604484, 16) and unique PatientIDs: 13109
2025-04-26 12:20:31,951 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (707219, 17) and unique PatientIDs: 2621
2025-04-26 12:20:32,228 - INFO - Successfully processed Vitals.csv file with final shape: (2626, 8) and unique PatientIDs: 2626


### Process Lab.csv

In [26]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = df,
                                 index_date_column = 'AdvancedDiagnosisDate',
                                 days_before = 90,
                                 days_after = 14,
                                 summary_lookback = 180)

2025-04-26 12:20:45,108 - INFO - Successfully read Lab.csv file with shape: (9373598, 17) and unique PatientIDs: 12700
2025-04-26 12:20:48,025 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (1888446, 18) and unique PatientIDs: 2540
2025-04-26 12:20:51,444 - INFO - Successfully processed Lab.csv file with final shape: (2626, 76) and unique PatientIDs: 2626


### Process MedicationAdministration.csv

In [27]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = df,
                                               index_date_column = 'AdvancedDiagnosisDate',
                                               days_before = 90,
                                               days_after = 0)

2025-04-26 12:20:52,695 - INFO - Successfully read MedicationAdministration.csv file with shape: (997836, 11) and unique PatientIDs: 10983
2025-04-26 12:20:52,969 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (196313, 12) and unique PatientIDs: 2234
2025-04-26 12:20:53,003 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (2626, 9) and unique PatientIDs: 2626


### Process Diagnosis.csv

In [28]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = df,
                                           index_date_column = 'AdvancedDiagnosisDate',
                                           days_before = None,
                                           days_after = 14)

2025-04-26 12:20:53,405 - INFO - Successfully read Diagnosis.csv file with shape: (625348, 6) and unique PatientIDs: 13129
2025-04-26 12:20:53,501 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (122914, 7) and unique PatientIDs: 2626
2025-04-26 12:20:53,842 - INFO - Successfully processed Diagnosis.csv file with final shape: (2626, 40) and unique PatientIDs: 2626


In [29]:
diagnosis_df['other_gi_met'] = (
    diagnosis_df['adrenal_met'] | diagnosis_df['peritoneum_met'] | diagnosis_df['gi_met']
)

diagnosis_df['other_combined_met'] = (
    diagnosis_df['brain_met'] | diagnosis_df['other_met']
)

diagnosis_df = diagnosis_df.drop(columns = ['adrenal_met', 'peritoneum_met', 'gi_met', 'brain_met', 'other_met'])

## Merge dataframes

In [30]:
final_df = merge_dataframes(demographics_df,
                            enhanced_df,
                            biomarkers_df,
                            ecog_df,
                            vitals_df,
                            labs_df,
                            medications_df,
                            diagnosis_df)

2025-04-26 12:20:53,854 - INFO - Anticipated number of merges: 7
2025-04-26 12:20:53,855 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 149
2025-04-26 12:20:53,856 - INFO - Dataset 1 shape: (2626, 6), unique PatientIDs: 2626
2025-04-26 12:20:53,857 - INFO - Dataset 2 shape: (2626, 13), unique PatientIDs: 2626
2025-04-26 12:20:53,858 - INFO - Dataset 3 shape: (2626, 4), unique PatientIDs: 2626
2025-04-26 12:20:53,858 - INFO - Dataset 4 shape: (2626, 3), unique PatientIDs: 2626
2025-04-26 12:20:53,859 - INFO - Dataset 5 shape: (2626, 8), unique PatientIDs: 2626
2025-04-26 12:20:53,859 - INFO - Dataset 6 shape: (2626, 76), unique PatientIDs: 2626
2025-04-26 12:20:53,860 - INFO - Dataset 7 shape: (2626, 9), unique PatientIDs: 2626
2025-04-26 12:20:53,861 - INFO - Dataset 8 shape: (2626, 37), unique PatientIDs: 2626
2025-04-26 12:20:53,867 - INFO - After merge 1 shape: (2626, 18), unique PatientIDs 2626
2025-04-26 12:20:53,86

In [31]:
final_df.shape

(2626, 149)

In [32]:
final_df.to_csv('../outputs/mUC_test_df.csv', index = False)

In [33]:
# Save dtypes
final_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/mUC_test_df_dtypes.csv')